# RAG assignment: baseline retrieval vs. Cross-Encoder re-ranking

This notebook compares two retrieval workflows on **one frozen PDF**: NIST SP 800-63B (June 2017, updated March 2020, with its withdrawal cover). This historical document is an experimental corpus, not current security advice.

**Baseline:** dense Top 10 → original Top 3 → Answer 1.  
**Re-ranking:** the same Top 10 → Cross-Encoder → new Top 3 → Answer 2.

All three models run locally; no API key or paid service is needed. Run cells from top to bottom. The first run downloads models; later runs reuse the cache. The notebook saves the corpus, every candidate/rank, actual LLM outputs, and timing samples under `outputs/`.

The stress-test set is deliberately exploratory, not a random benchmark. We retain its failures and successes. The assignment report must be written by the student in their own words; this notebook and the report worksheet provide evidence, not a submission-ready report.

## 1. Environment and fixed configuration

Install `requirements.txt` in a virtual environment before opening the notebook. CPU works everywhere; CUDA or Apple MPS is used for answer generation if available. Retrieval runs on CPU with four threads for consistent small-batch timing. Model revisions and the PDF checksum are fixed, so changing the source or models is an explicit experiment change.

In [1]:
from pathlib import Path
import os

ROOT = Path.cwd()
assert (ROOT / "RAG_LMA_Assignment_v1.pdf").exists(), "Run this notebook from the repository root."
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import gc
import hashlib
import importlib.metadata
import json
import platform
import random
import re
import statistics
from datetime import datetime
from time import perf_counter
from zoneinfo import ZoneInfo
import torch  # Load PyTorch's OpenMP runtime before FAISS on macOS.
import faiss
import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
faiss.omp_set_num_threads(4)
DATA = ROOT / "data"
OUT = ROOT / "outputs"
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

CONFIG = {
    "pdf_url": "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-63b.pdf",
    "pdf_sha256": "ccfce7510a1267933f912d023306c6e7d485f21be63d271108b61aad6139127e",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "reranker_revision": "233902d25c440f23af6f7d6e94d2946bac0bee0a",
    "answer_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "answer_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    "max_chunk_tokens": 220,
    "overlap_tokens": 40,
    "retrieve_k": 10,
    "context_k": 3,
    "timing_repeats": 5,
    "max_new_tokens": 384,
    "seed": SEED,
}

def save_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")

LLM_DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Retrieval device: cpu | Answer model device:", LLM_DEVICE)
print("Outputs:", OUT.name)

Retrieval device: cpu | Answer model device: mps
Outputs: outputs


## 2. Source PDF and extraction

The PDF is included so the document cannot silently change. We verify its SHA-256 before use. Physical PDF pages are **one-based**; the printed document page is also retained. The first 11 PDF pages contain the withdrawal notice, cover, and front matter; the knowledge corpus starts at PDF page 12 (printed page 1). We keep all remaining pages, including the informative sections and appendix, because they are realistic retrieval distractors.

Cleaning removes only the repeated running header, availability notice, and isolated page-number line. Extracted text remains inspectable in `outputs/pages.json`.

In [2]:
pdf_path = DATA / "nist_sp_800_63b.pdf"
if not pdf_path.exists():
    response = requests.get(CONFIG["pdf_url"], timeout=120)
    response.raise_for_status()
    pdf_path.write_bytes(response.content)
actual_sha = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
assert actual_sha == CONFIG["pdf_sha256"], "Source PDF changed; restore the frozen file or explicitly revise the experiment."
reader = PdfReader(pdf_path)
assert len(reader.pages) == 80

def clean_page(text):
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line and not (
        line.startswith("NIST SP 800-63B")
        or line == "AUTHENTICATION & LIFECYCLE MANAGEMENT"
        or line.startswith("This publication is available free of charge from:")
        or re.fullmatch(r"\d+", line)
    )]
    return re.sub(r"\s+", " ", " ".join(lines)).strip()

pages = [
    {"pdf_page": i+1, "printed_page": i-10, "text": clean_page(reader.pages[i].extract_text() or "")}
    for i in range(11, len(reader.pages))
]
assert all(page["text"] for page in pages), "Extraction produced an empty body page."
save_json("pages.json", pages)
save_json("document.json", {
    "title": "NIST SP 800-63B: Digital Identity Guidelines — Authentication and Lifecycle Management",
    "edition": "June 2017; updates through 2020-03-02; withdrawal cover added 2025",
    "status": "Withdrawn 2025-08-01; historical corpus only",
    "source_url": CONFIG["pdf_url"], "sha256": actual_sha,
    "total_pdf_pages": len(reader.pages), "indexed_pdf_pages": [12, 80],
    "page_number_note": "Physical PDF page = printed Arabic page + 11 for body pages.",
})
print(f"Verified {len(reader.pages)} PDF pages; extracted {len(pages)} body pages.")
display(Markdown("**Extraction sample — PDF page 24 / printed page 13**\n\n" + pages[12]["text"][:1500]))

Verified 80 PDF pages; extracted 69 body pages.


**Extraction sample — PDF page 24 / printed page 13**

5 Authenticator and Verifier Requirements This section is normative. This section provides the detailed requirements specific to each type of authenticator. With the exception of reauthentication requirements specified in Section 4 and the requirement for verifier impersonation resistance at AAL3 described in Section 5.2.5, the technical requirements for each of the authenticator types are the same regardless of the AAL at which the authenticator is used. Requirements by Authenticator Type 5.1.1 Memorized Secrets A Memorized Secret authenticator — commonly referred to as a password or, if numeric, a PIN — is a secret value intended to be chosen and memorized by the user. Memorized secrets need to be of sufficient complexity and secrecy that it would be impractical for an attacker to guess or otherwise discover the correct secret value. A memorized secret is something you know. 5.1.1.1 Memorized Secret Authenticators Memorized secrets SHALL be at least 8 characters in length if chosen by the subscriber. Memorized secrets chosen randomly by the CSP or verifier SHALL be at least 6 characters in length and MAY be entirely numeric. If the CSP or verifier disallows a chosen memorized secret based on its appearance on a blacklist of compromised values, the subscriber SHALL be required to choose a different memorized secret. No other complexity requirements for memorized secrets SHOULD be imposed. A rationale for this is presented in Appendix A Strength of Memorized Secrets. 5.1.1.2 

## 3. Load retrieval models and freeze the chunk corpus

The Bi-Encoder creates one vector per passage independently of the question. The Cross-Encoder later sees each question and passage together. Its scores are used only to sort its ten input candidates; they are not directly comparable to cosine similarities.

Chunks stay within a page, pack complete sentences when possible, and retain up to 40 tokens of sentence overlap. A sentence exceeding the chunk budget is split at word boundaries within the token budget. The 220-token budget fits the embedding model's 256-token limit and leaves ample room for a question in the Cross-Encoder's 512-token input. Token limits are checked instead of silently truncating evidence.

In [3]:
setup_times = {}
t0 = perf_counter()
embedder = SentenceTransformer(CONFIG["embedding_model"], revision=CONFIG["embedding_revision"], device="cpu")
reranker = CrossEncoder(CONFIG["reranker_model"], revision=CONFIG["reranker_revision"], device="cpu", max_length=512)
setup_times["retrieval_model_loading_s"] = perf_counter() - t0

def token_count(text):
    return len(embedder.tokenizer.encode(text, add_special_tokens=False, verbose=False))

def chunk_page(page):
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9•])", page["text"])
    units = []
    for sentence in sentences:
        ids = embedder.tokenizer.encode(sentence, add_special_tokens=False, verbose=False)
        if len(ids) <= CONFIG["max_chunk_tokens"]:
            units.append(sentence)
        else:
            # Keep original words: decoding WordPiece windows and re-encoding can
            # change token counts and corrupt exact source phrases.
            window = []
            for word in sentence.split():
                if window and token_count(" ".join(window + [word])) > CONFIG["max_chunk_tokens"]:
                    units.append(" ".join(window))
                    overlap = []
                    for previous in reversed(window):
                        if token_count(" ".join([previous] + overlap)) > CONFIG["overlap_tokens"]:
                            break
                        overlap.insert(0, previous)
                    window = overlap
                window.append(word)
            if window:
                units.append(" ".join(window))
    groups, current = [], []
    for unit in units:
        if current and token_count(" ".join(current + [unit])) > CONFIG["max_chunk_tokens"]:
            groups.append(" ".join(current))
            overlap = []
            for previous in reversed(current):
                if token_count(" ".join([previous] + overlap)) > CONFIG["overlap_tokens"]:
                    break
                overlap.insert(0, previous)
            current = overlap
            while current and token_count(" ".join(current + [unit])) > CONFIG["max_chunk_tokens"]:
                current.pop(0)
        current.append(unit)
    if current:
        groups.append(" ".join(current))
    return [dict(chunk_id=f"p{page['pdf_page']:03d}_c{j:02d}", pdf_page=page["pdf_page"],
                 printed_page=page["printed_page"], text=text, token_count=token_count(text))
            for j, text in enumerate(groups, 1)]

chunks = [chunk for page in pages for chunk in chunk_page(page)]
assert len(chunks) > CONFIG["retrieve_k"]
assert max(c["token_count"] for c in chunks) <= CONFIG["max_chunk_tokens"]
assert max(c["token_count"] for c in chunks) + 2 <= embedder.max_seq_length
assert len({c["chunk_id"] for c in chunks}) == len(chunks)
chunk_by_id = {c["chunk_id"]: c for c in chunks}
save_json("chunks.json", chunks)
corpus_sha = hashlib.sha256(json.dumps(chunks, sort_keys=True).encode()).hexdigest()
print(f"Frozen corpus: {len(chunks)} chunks; longest = {max(c['token_count'] for c in chunks)} tokens")
print("Corpus SHA-256:", corpus_sha)
display(pd.DataFrame(chunks)[["chunk_id", "pdf_page", "printed_page", "token_count"]].head(10))

Frozen corpus: 215 chunks; longest = 220 tokens
Corpus SHA-256: cf0d4aaca28fec357fc2b7688c2328504322e461c05b140cd66b99fab4803d3f


,chunk_id,pdf_page,printed_page,token_count
0,p012_c01,12,1,49
1,p013_c01,13,2,209
2,p013_c02,13,2,211
3,p013_c03,13,2,197
4,p013_c04,13,2,86
5,p014_c01,14,3,212
6,p014_c02,14,3,169
7,p015_c01,15,4,23
8,p016_c01,16,5,217
9,p016_c02,16,5,163


## 4. Dense vector store and the two retrieval workflows

We L2-normalize document and question vectors. FAISS `IndexFlatIP` then returns exact inner-product search, which equals cosine similarity on normalized vectors. A small exact index avoids approximate-search variability.

`rerank()` accepts only the candidate list produced by `retrieve()` and preserves the original rank. Both workflows select exactly three passages.

In [4]:
t0 = perf_counter()
vectors = embedder.encode([c["text"] for c in chunks], batch_size=32, normalize_embeddings=True, show_progress_bar=False)
vectors = np.ascontiguousarray(vectors, dtype=np.float32)
assert np.isfinite(vectors).all()
assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
setup_times["embedding_and_indexing_s"] = perf_counter() - t0
np.save(OUT / "embeddings.npy", vectors)
faiss.write_index(index, str(OUT / "index.faiss"))

def retrieve(question):
    query_vector = embedder.encode([question], normalize_embeddings=True, show_progress_bar=False)
    scores, indices = index.search(np.ascontiguousarray(query_vector, dtype=np.float32), CONFIG["retrieve_k"])
    return [dict(chunks[int(i)], dense_score=float(score), baseline_rank=rank)
            for rank, (i, score) in enumerate(zip(indices[0], scores[0]), 1)]

def rerank(question, candidates):
    assert len(candidates) == CONFIG["retrieve_k"]
    pairs = [(question, c["text"]) for c in candidates]
    pair_lengths = [len(reranker.tokenizer(q, passage, truncation=False)["input_ids"]) for q, passage in pairs]
    assert max(pair_lengths) <= 512, "A query/passage pair would be truncated."
    scores = reranker.predict(pairs, batch_size=10, show_progress_bar=False, activation_fn=torch.nn.Identity())
    scored = [dict(c, cross_encoder_score=float(score)) for c, score in zip(candidates, scores)]
    ranked = sorted(scored, key=lambda c: (-c["cross_encoder_score"], c["baseline_rank"]))
    result = [dict(c, reranked_rank=rank) for rank, c in enumerate(ranked, 1)]
    assert {c["chunk_id"] for c in result} == {c["chunk_id"] for c in candidates}
    return result

print(f"FAISS index: {index.ntotal} vectors × {index.d} dimensions")
print("One-time setup:", setup_times)

FAISS index: 215 vectors × 384 dimensions
One-time setup: {'retrieval_model_loading_s': 0.12075025000012829, 'embedding_and_indexing_s': 1.625738167000236}


## 5. Source-grounded exploratory stress tests

Each question has a page and an exact critical phrase identified from the PDF **before retrieval**. That phrase labels the important evidence; it is never passed into retrieval, re-ranking, or the answer prompt. Multiple overlapping chunks can contain the same phrase, so we use the best rank among them.

These are **critical-evidence ranks**, not an exhaustive relevance annotation. Other chunks may contain equivalent evidence. We inspect the final contexts and answers before making a qualitative claim. A missing critical phrase outside the Top 10 is recorded as a candidate-recall failure; the Cross-Encoder cannot recover it.

In [5]:
# (id, category, question, physical PDF page, critical source phrase, expected answer checklist)
query_specs = [
("q01", "keyword_trap", "Can a password reset hint be shown before someone signs in?", 25, "hint", "No hint accessible to an unauthenticated claimant; do not confuse password hints with strength guidance."),
("q02", "exceptions", "Do users need to change their password every month even when it has not leaked?", 25, "However, verifiers SHALL force a change", "No arbitrary periodic changes; force a change when compromise is evidenced."),
("q03", "exceptions", "Does the ban on scheduled password changes mean a stolen password can be kept?", 25, "However, verifiers SHALL force a change", "No; evidence of compromise requires a change despite the advice against arbitrary rotation."),
("q04", "document_specific", "Can a six-digit PIN generated by the service be acceptable even though user-selected passwords require eight characters?", 24, "Memorized secrets chosen randomly by the CSP", "Subscriber-chosen: at least 8; randomly assigned by CSP/verifier: at least 6 and may be numeric."),
("q05", "negation", "May a verifier shorten an overlong password instead of checking the full value?", 24, "Truncation of the secret SHALL NOT be performed", "Truncation is prohibited; do not confuse it with permitted normalization."),
("q06", "exceptions", "Can several blank spaces in a passphrase be collapsed, and what length must remain?", 24, "provided that the result is at least 8 characters", "Consecutive spaces may become one only if the result is at least 8 characters; no truncation."),
("q07", "keyword_trap", "Is copying a password from a password manager into the login form permitted?", 25, "paste", "Verifiers SHOULD permit paste to support password managers."),
("q08", "negation", "Can an email inbox or internet phone call serve as the second authentication channel?", 28, "Methods that do not prove possession of a specific device", "Email and VOIP SHALL NOT be used for out-of-band authentication because they do not prove possession of a specific device."),
("q09", "negation", "Does a locked phone need to reveal the one-time login code on its screen?", 28, "SHOULD NOT display the authentication secret", "It SHOULD NOT display the secret while locked, but SHOULD indicate receipt."),
("q10", "exceptions", "Can the same out-of-band code be used twice if its ten-minute lifetime has not ended?", 29, "only once during the validity period", "No; accept a given authentication secret only once even within its validity period."),
("q11", "document_specific", "What is the deadline for finishing an out-of-band authentication transaction?", 29, "invalid if not completed within 10 minutes", "The authentication is invalid if not completed within 10 minutes."),
("q12", "exceptions", "When biometrics have liveness detection, does the usual five-failure limit still apply?", 38, "or 10 consecutive failed attempts if PAD", "Maximum 5 consecutive failures, or 10 if qualifying PAD is implemented; then delay or disable as specified."),
("q13", "exceptions", "Does the general limit of 100 wrong guesses also allow 100 failed fingerprint scans?", 38, "or 10 consecutive failed attempts if PAD", "No; biometric-specific limits are 5, or 10 with qualifying PAD, followed by specified mitigation."),
("q14", "negation", "Is a fingerprint alone a complete authenticator?", 38, "Biometrics SHALL be used only as part of multi-factor", "Biometrics require multi-factor authentication with a physical authenticator."),
("q15", "keyword_trap", "If a camera sees my face, does that prove I intended to log in?", 41, "claimant’s face using a camera normally would not by itself", "No; face observation normally does not itself establish intent, unlike deliberate fingerprint presentation."),
("q16", "negation", "If an OAuth refresh token still works, does that prove the user is still present?", 49, "SHALL NOT be interpreted by the RP as presence", "No; access and refresh tokens may outlive the authentication session and user presence."),
("q17", "negation", "May an authenticated website switch from HTTPS to HTTP after login?", 49, "Authenticated sessions SHALL NOT fall back", "No fallback to an insecure transport after authentication."),
("q18", "document_specific", "Can a login session survive restarting the app or rebooting the device?", 50, "Session secrets SHALL be non-persistent", "No; session secrets SHALL NOT survive application restart or device reboot."),
("q19", "exceptions", "At AAL2, can possession of a hardware key alone renew an existing session?", 50, "not a physical authenticator", "A memorized secret or biometric is required; the session secret already supplies the possession factor."),
("q20", "keyword_trap", "Is setting the browser cookie expiry sufficient to enforce session timeout?", 49, "SHALL NOT be depended upon to enforce session timeouts", "No; cookie expiry should follow session validity but must not be relied on to enforce timeout."),
("q21", "document_specific", "How is the length of a password containing Unicode characters counted?", 24, "each Unicode code point SHALL be counted", "Count each Unicode code point as one character, not each byte."),
("q22", "keyword_trap", "Should a password verifier require at least one capital letter, one digit, and one symbol?", 25, "SHOULD NOT impose other composition rules", "SHOULD NOT impose composition rules; distinguish these from blocklist checks."),
("q23", "negation", "Can a memorized password resist replay merely because it is sent over TLS?", 41, "memorized secrets are not considered replay resistant", "No; the secret itself is reused; protected-channel replay resistance is distinct."),
("q24", "document_specific", "What happens after too many failed biometric attempts when no alternate factor is available?", 38, "Impose a delay of at least 30 seconds", "One specified option is an exponentially increasing delay starting at 30 seconds; disabling requires an available alternative."),
("q25", "keyword_trap", "Are a user's face and fingerprints secret in the same way as a password?", 37, "Biometric characteristics do not constitute secrets", "No; biometric characteristics can be captured or obtained without the user's knowledge."),
("q26", "exceptions", "Does a successful login clear earlier failed attempts from the same IP address?", 37, "SHOULD disregard any previous failed attempts", "The verifier SHOULD disregard prior failures for that user from the same IP after successful authentication."),
("q27", "negation", "May a one-time password be accepted again while it remains valid?", 31, "accept a given time-based OTP only once", "No; a valid OTP is accepted only once to provide replay resistance."),
("q28", "keyword_trap", "Can users be asked for their first pet's name when selecting a login password?", 25, "SHALL NOT prompt subscribers to use specific types", "No prompting for specific personal information such as first-pet names."),
("q29", "exceptions", "Can a passphrase with repeated spaces be normalized even though truncation is prohibited?", 24, "provided that the result is at least 8 characters", "Space collapsing is permitted provided the result is at least 8 characters; truncation remains prohibited."),
("q30", "document_specific", "Does a federation assertion guarantee that the identity provider and relying party sessions expire together?", 50, "SHALL NOT be any assumption of correlation", "No; CSP and RP sessions are managed separately and must not be assumed correlated."),
("q31", "negation", "Is central biometric matching preferred over checking on the user's device?", 38, "local comparison is preferred", "No; local comparison is preferred because centralized attacks can scale further."),
("q32", "keyword_trap", "Must passwords be hidden by dots at all times while the claimant enters them?", 25, "offer an option to display the secret", "The verifier SHOULD offer a display option in suitable conditions, and MAY briefly show individual characters."),
]

def normalized(text):
    return re.sub(r"\s+", " ", text).casefold()

queries = []
for qid, category, question, page, anchor, expected in query_specs:
    gold_ids = [c["chunk_id"] for c in chunks if c["pdf_page"] == page and normalized(anchor) in normalized(c["text"])]
    assert gold_ids, f"Gold phrase missing after chunking: {qid} / {anchor}"
    queries.append(dict(query_id=qid, category=category, question=question, gold_pdf_page=page,
                        critical_phrase=anchor, gold_chunk_ids=gold_ids, expected_answer=expected))
save_json("queries.json", queries)
print(f"{len(queries)} source-grounded exploratory queries; all critical phrases verified.")
display(pd.DataFrame(queries)[["query_id", "category", "question", "gold_pdf_page"]])

32 source-grounded exploratory queries; all critical phrases verified.


,query_id,category,question,gold_pdf_page
0,q01,keyword_trap,Can a password reset hint be shown before some...,25
1,q02,exceptions,Do users need to change their password every m...,25
2,q03,exceptions,Does the ban on scheduled password changes mea...,25
3,q04,document_specific,Can a six-digit PIN generated by the service b...,24
4,q05,negation,May a verifier shorten an overlong password in...,24
5,q06,exceptions,Can several blank spaces in a passphrase be co...,24
6,q07,keyword_trap,Is copying a password from a password manager ...,25
7,q08,negation,Can an email inbox or internet phone call serv...,28
8,q09,negation,Does a locked phone need to reveal the one-tim...,28
9,q10,exceptions,Can the same out-of-band code be used twice if...,29


## 6. Explore rank shifts without generating answers yet

We retain all ten candidates and both ranks for every question. `promoted_into_top3` means the labeled critical evidence moved from ranks 4–10 into ranks 1–3. This is a retrieval event, not an automatic claim that the generated answer improved. We separately record regressions and evidence absent from the ten candidates.

In [6]:
explorations, rank_rows = [], []
for query in queries:
    dense = retrieve(query["question"])
    ranked = rerank(query["question"], dense)
    gold = set(query["gold_chunk_ids"])
    before = min((c["baseline_rank"] for c in dense if c["chunk_id"] in gold), default=None)
    after = min((c["reranked_rank"] for c in ranked if c["chunk_id"] in gold), default=None)
    row = dict(query_id=query["query_id"], category=query["category"], question=query["question"],
               gold_pdf_page=query["gold_pdf_page"], baseline_rank=before, reranked_rank=after,
               promoted_into_top3=before is not None and before > 3 and after <= 3,
               demoted_out_of_top3=before is not None and before <= 3 and after > 3,
               missing_from_top10=before is None)
    explorations.append(dict(query=query, summary=row, dense=dense, reranked=ranked))
    for c in ranked:
        rank_rows.append(dict(query_id=query["query_id"], chunk_id=c["chunk_id"], pdf_page=c["pdf_page"],
                              baseline_rank=c["baseline_rank"], reranked_rank=c["reranked_rank"],
                              dense_score=c["dense_score"], cross_encoder_score=c["cross_encoder_score"],
                              critical_evidence=c["chunk_id"] in gold))
rank_summary = pd.DataFrame([e["summary"] for e in explorations])
rank_summary.to_csv(OUT / "rank_summary.csv", index=False)
pd.DataFrame(rank_rows).to_csv(OUT / "candidate_ranks.csv", index=False)
save_json("exploration.json", explorations)
display(rank_summary[["query_id", "category", "baseline_rank", "reranked_rank", "promoted_into_top3", "missing_from_top10"]])
print("Critical-evidence promotions:", int(rank_summary.promoted_into_top3.sum()))
print("Critical-evidence regressions:", int(rank_summary.demoted_out_of_top3.sum()))
print("Critical evidence missing from Top 10:", int(rank_summary.missing_from_top10.sum()))

,query_id,category,baseline_rank,reranked_rank,promoted_into_top3,missing_from_top10
0,q01,keyword_trap,NaN,NaN,False,True
1,q02,exceptions,7.0,3.0,True,False
2,q03,exceptions,NaN,NaN,False,True
3,q04,document_specific,NaN,NaN,False,True
4,q05,negation,10.0,6.0,False,False
5,q06,exceptions,6.0,1.0,True,False
6,q07,keyword_trap,NaN,NaN,False,True
7,q08,negation,NaN,NaN,False,True
8,q09,negation,3.0,1.0,False,False
9,q10,exceptions,3.0,1.0,False,False


Critical-evidence promotions: 6
Critical-evidence regressions: 1
Critical evidence missing from Top 10: 7


## 7. Choose transparent case studies and measure retrieval latency

Choose one observed promotion per category where available, then enough further promotions for up to six positive cases. Include up to two non-improving controls. Selection is based on the recorded exploratory results and is intentionally disclosed; it is not an unbiased accuracy estimate.

Timing excludes model loading, indexing, file writes, and LLM generation. After a warm-up, each selected question is retrieved and re-ranked five times. Dense time includes question embedding and FAISS search. Total re-ranking retrieval time includes **that same dense retrieval plus Cross-Encoder scoring and sorting**. We save raw samples, then summarize with medians. These timings describe this machine and configuration.

In [7]:
selected = []
for category in ["keyword_trap", "exceptions", "negation", "document_specific"]:
    match = next((e for e in explorations if e["summary"]["promoted_into_top3"] and e["query"]["category"] == category), None)
    if match is not None:
        selected.append(match)
for e in explorations:
    if len(selected) >= 6:
        break
    if e["summary"]["promoted_into_top3"] and e not in selected:
        selected.append(e)
controls = [e for e in explorations if not e["summary"]["promoted_into_top3"]]
# Prefer a regression or candidate-recall failure as one informative control.
control = next((e for e in controls if e["summary"]["demoted_out_of_top3"]), None)
if control is None:
    control = next((e for e in controls if e["summary"]["missing_from_top10"]), None)
if control is not None:
    selected.append(control)
control = next((e for e in controls if e not in selected and e["summary"]["baseline_rank"] is not None
                and e["summary"]["baseline_rank"] <= 3 and e["summary"]["reranked_rank"] <= 3), None)
if control is not None:
    selected.append(control)
selected_ids = [e["query"]["query_id"] for e in selected]
save_json("selected_cases.json", selected_ids)
print("Selected cases:", selected_ids)

_ = rerank(queries[0]["question"], retrieve(queries[0]["question"]))
timing_samples = []
for e in selected:
    q = e["query"]
    for repeat in range(CONFIG["timing_repeats"]):
        t0 = perf_counter()
        dense = retrieve(q["question"])
        t1 = perf_counter()
        ranked = rerank(q["question"], dense)
        t2 = perf_counter()
        timing_samples.append(dict(query_id=q["query_id"], repeat=repeat+1,
                                   baseline_retrieval_ms=(t1-t0)*1000,
                                   cross_encoder_ms=(t2-t1)*1000,
                                   reranked_retrieval_ms=(t2-t0)*1000))
timing_frame = pd.DataFrame(timing_samples)
timing_frame.to_csv(OUT / "latency_samples.csv", index=False)
latency_summary = timing_frame.groupby("query_id", sort=False).median(numeric_only=True).drop(columns="repeat").reset_index()
latency_summary.to_csv(OUT / "latency_summary.csv", index=False)
display(latency_summary.round(2))

Selected cases: ['q28', 'q02', 'q11', 'q06', 'q13', 'q29', 'q25', 'q09']


,query_id,baseline_retrieval_ms,cross_encoder_ms,reranked_retrieval_ms
0,q28,5.43,85.83,91.35
1,q02,5.35,86.19,91.53
2,q11,5.53,89.30,94.83
3,q06,5.35,86.89,92.54
4,q13,5.63,84.23,89.75
5,q29,5.75,92.04,98.17
6,q25,5.75,96.38,101.68
7,q09,5.63,88.03,93.22


## 8. Generate both answers with one identical prompt

The same local instruction-tuned LLM receives exactly three passages in the same formatting. Greedy decoding (`do_sample=False`) and a fixed token limit are used for both answers. Neither the expected answer, critical phrase, relevance scores, nor pipeline name is shown to the model.

The model can still make errors even with the right evidence. We preserve its raw response, prompt, token counts, finish reason, and elapsed time. An answer that reaches the token limit is explicitly labeled. Contexts are never silently truncated.

In [8]:
t0 = perf_counter()
llm_tokenizer = AutoTokenizer.from_pretrained(CONFIG["answer_model"], revision=CONFIG["answer_revision"])
llm_dtype = torch.float16 if LLM_DEVICE in {"mps", "cuda"} else torch.float32
llm = AutoModelForCausalLM.from_pretrained(CONFIG["answer_model"], revision=CONFIG["answer_revision"], torch_dtype=llm_dtype)
llm.to(LLM_DEVICE).eval()
setup_times["answer_model_loading_s"] = perf_counter() - t0

SYSTEM_PROMPT = """You are a careful reading assistant. Answer only from the provided historical NIST passages.
Answer the specific question in 2 or 3 sentences, at most 80 words. Start with the direct answer.
Use the specific rule for the question rather than an unrelated general rule. Include any relevant
exception, numeric condition, or prohibition. Preserve the meaning of SHALL, SHOULD, and MAY.
Cite at least one supporting passage using its exact chunk ID in square brackets.
If the passages do not answer the question, say the supplied passages are insufficient.
Do not invent facts or describe unrelated procedures."""

def make_messages(question, context):
    assert len(context) == CONFIG["context_k"]
    evidence = "\n\n".join(f"[{c['chunk_id']}] PDF page {c['pdf_page']} (printed page {c['printed_page']}):\n{c['text']}" for c in context)
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Passages:\n{evidence}\n\nQuestion: {question}\nAnswer:"}]

def synchronize():
    if LLM_DEVICE == "cuda":
        torch.cuda.synchronize()
    elif LLM_DEVICE == "mps":
        torch.mps.synchronize()

def generate_answer(question, context):
    messages = make_messages(question, context)
    # Synchronize accelerator work at both timer boundaries.
    synchronize()
    t0 = perf_counter()
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(text, add_special_tokens=False, return_tensors="pt").to(LLM_DEVICE)
    input_tokens = int(inputs["input_ids"].shape[-1])
    assert input_tokens + CONFIG["max_new_tokens"] <= llm.config.max_position_embeddings
    with torch.inference_mode():
        generated = llm.generate(**inputs, do_sample=False, max_new_tokens=CONFIG["max_new_tokens"],
                                 pad_token_id=llm_tokenizer.eos_token_id)
    synchronize()
    elapsed_ms = (perf_counter()-t0)*1000
    new_ids = generated[0, input_tokens:]
    answer = llm_tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return dict(answer=answer, generation_ms=elapsed_ms, input_tokens=input_tokens,
                output_tokens=len(new_ids), hit_token_limit=len(new_ids) >= CONFIG["max_new_tokens"],
                messages=messages, context_ids=[c["chunk_id"] for c in context])

# One short generation warm-up, excluded from per-answer timing.
warm = llm_tokenizer("Answer briefly: hello", return_tensors="pt").to(LLM_DEVICE)
with torch.inference_mode():
    _ = llm.generate(**warm, do_sample=False, max_new_tokens=8, pad_token_id=llm_tokenizer.eos_token_id)
synchronize()
print("Answer model ready:", CONFIG["answer_model"], LLM_DEVICE)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Answer model ready: Qwen/Qwen2.5-1.5B-Instruct mps


In [9]:
answers = []
for case_index, e in enumerate(selected):
    q = e["query"]
    contexts = {"baseline": e["dense"][:3], "reranked": e["reranked"][:3]}
    generated = {}
    # Alternate order so one pipeline is not always the first measured generation.
    order = ["baseline", "reranked"] if case_index % 2 == 0 else ["reranked", "baseline"]
    for pipeline in order:
        generated[pipeline] = generate_answer(q["question"], contexts[pipeline])
        print(q["query_id"], pipeline, f"{generated[pipeline]['generation_ms']/1000:.1f}s", flush=True)
    result = dict(query=q, rank_summary=e["summary"], baseline=generated["baseline"], reranked=generated["reranked"])
    answers.append(result)
    save_json("answers.json", answers)  # Save real progress after every completed pair.
    display(Markdown(f"### {q['query_id']}: {q['question']}\n\n"
                     f"**Expected evidence:** PDF page {q['gold_pdf_page']}; {q['expected_answer']}\n\n"
                     f"**Answer 1 — baseline:** {generated['baseline']['answer']}\n\n"
                     f"**Answer 2 — re-ranked:** {generated['reranked']['answer']}"))

q28 baseline 3.8s


q28 reranked 2.9s


### q28: Can users be asked for their first pet's name when selecting a login password?

**Expected evidence:** PDF page 25; No prompting for specific personal information such as first-pet names.

**Answer 1 — baseline:** No, users cannot be asked for their first pet's name when selecting a login password according to the passage. The text states that "users' password choices are very predictable" and mentions examples like "Password1!" and "Password1!", indicating that common patterns exist and suggest avoiding certain inputs. Therefore, asking for personal details like a pet's name could lead to predictable passwords being chosen.

**Answer 2 — re-ranked:** No, according to the passage, users shall not be prompted to use specific types of information when choosing memorized secrets, including asking about their first pet's name. The passage states "Verifiers SHALL NOT prompt subscribers to use specific types of information" when choosing memorized secrets.

q02 reranked 2.5s


q02 baseline 4.3s


...

q25 reranked 3.9s


### q25: Are a user's face and fingerprints secret in the same way as a password?

**Expected evidence:** PDF page 37; No; biometric characteristics can be captured or obtained without the user's knowledge.

**Answer 1 — baseline:** No, a user's face and fingerprints are not considered secrets in the same way as a password. According to the passage [p037_c03], "Biometric characteristics do not constitute secrets." This means that unlike passwords, which are typically kept secret and protected, biometric data like faces and fingerprints can be easily obtained through various means such as photographs or scans.

**Answer 2 — re-ranked:** No, a user's face and fingerprints are not considered secrets in the same way as a password according to the given passages. Passwords are typically memorized and subject to change, while faces and fingerprints are biometric identifiers that are unique to individuals and cannot be forgotten or changed. The passages emphasize the limitations of human memory in remembering complex secrets and suggest alternative methods like password managers to enhance security without requiring constant changes.

q09 reranked 4.0s


q09 baseline 2.9s


### q09: Does a locked phone need to reveal the one-time login code on its screen?

**Expected evidence:** PDF page 28; It SHOULD NOT display the secret while locked, but SHOULD indicate receipt.

**Answer 1 — baseline:** No, according to the passage [p020_c05], when a device like a smartphone is used in the authentication process, the unlocking of that device shall not be considered one of the authentication factors. Therefore, a locked phone does not need to reveal the one-time login code on its screen.

**Answer 2 — re-ranked:** A locked phone does not need to reveal the one-time login code on its screen according to the first passage [p028_c04]. The passage states that if a secret is sent by the verifier to the out-of-band device, the device SHOULD NOT display the authentication secret while it is locked by the owner. Therefore, even though the phone might be locked, the secret itself would not appear on the screen.

## 9. Comparison artifacts and validation

The tables below link retrieval events to the actual generated answers. End-to-end totals combine each question's median retrieval latency with its single measured generation call; they are labeled estimates rather than repeated full-pipeline measurements. LLM generation can dominate overall runtime, so retrieval latency is reported separately.

Automated checks verify candidate reuse, prompt equality, source traceability, vector normalization, non-truncation, and timing arithmetic. They cannot replace reading the answers. `outputs/qualitative_review.json`, when present, contains a separate manual inspection of the final outputs; the report worksheet asks the student to verify these observations against the PDF.

In [10]:
comparison_rows = []
for result in answers:
    q = result["query"]
    timing = latency_summary.set_index("query_id").loc[q["query_id"]]
    comparison_rows.append(dict(query_id=q["query_id"], category=q["category"], question=q["question"],
        critical_pdf_page=q["gold_pdf_page"], baseline_rank=result["rank_summary"]["baseline_rank"],
        reranked_rank=result["rank_summary"]["reranked_rank"],
        promoted_into_top3=result["rank_summary"]["promoted_into_top3"],
        baseline_context=", ".join(result["baseline"]["context_ids"]),
        reranked_context=", ".join(result["reranked"]["context_ids"]),
        answer_1=result["baseline"]["answer"], answer_2=result["reranked"]["answer"],
        baseline_retrieval_ms=float(timing["baseline_retrieval_ms"]),
        reranked_retrieval_ms=float(timing["reranked_retrieval_ms"]),
        answer_1_generation_ms=result["baseline"]["generation_ms"],
        answer_2_generation_ms=result["reranked"]["generation_ms"],
        baseline_total_estimate_ms=float(timing["baseline_retrieval_ms"])+result["baseline"]["generation_ms"],
        reranked_total_estimate_ms=float(timing["reranked_retrieval_ms"])+result["reranked"]["generation_ms"]))
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUT / "comparison.csv", index=False)

assert np.allclose(timing_frame.baseline_retrieval_ms + timing_frame.cross_encoder_ms,
                   timing_frame.reranked_retrieval_ms)
for e in explorations:
    assert len(e["dense"]) == len(e["reranked"]) == 10
    assert {c["chunk_id"] for c in e["dense"]} == {c["chunk_id"] for c in e["reranked"]}
    assert sorted(c["baseline_rank"] for c in e["dense"]) == list(range(1, 11))
for result in answers:
    for pipeline in ["baseline", "reranked"]:
        record = result[pipeline]
        assert len(record["context_ids"]) == len(set(record["context_ids"])) == 3
        assert all(chunk_id in chunk_by_id for chunk_id in record["context_ids"])
        assert record["answer"] and not record["hit_token_limit"], "An answer is empty or cut off; inspect and rerun."
        assert record["messages"][0]["content"] == SYSTEM_PROMPT
        assert record["messages"][-1]["content"].endswith(f"Question: {result['query']['question']}\nAnswer:")
assert rank_summary.promoted_into_top3.sum() >= 2, "Need at least two verified retrieval promotions; retain results and extend exploration."

versions = {}
for package in ["torch", "transformers", "sentence-transformers", "huggingface-hub", "numpy", "faiss-cpu", "pypdf", "pandas", "requests", "nbformat", "nbclient", "ipykernel"]:
    versions[package] = importlib.metadata.version(package)
save_json("environment.json", dict(python=platform.python_version(), platform=platform.platform(),
    machine=platform.machine(), llm_device=LLM_DEVICE, retrieval_device="cpu", torch_threads=4,
    versions=versions, config=CONFIG, corpus_sha256=corpus_sha, setup_times_s=setup_times,
    executed_at=datetime.now(ZoneInfo("Asia/Kolkata")).isoformat(),
    generation_settings={"do_sample": False, "max_new_tokens": CONFIG["max_new_tokens"], "dtype": str(llm_dtype)},
    timing_note="Five warmed retrieval repeats/query; LLM calls measured once; end-to-end values are estimates."))
(OUT / "requirements-tested.txt").write_text("\n".join(f"{name}=={version}" for name, version in versions.items()) + "\n")

# Plain Markdown evidence is convenient to review without a notebook renderer.
evidence = ["# Recorded experiment evidence", "", "Generated from actual notebook outputs. This is not the student report.", ""]
for result in answers:
    q = result["query"]
    evidence += [f"## {q['query_id']}: {q['question']}", "",
        f"Category: {q['category']}; critical PDF page: {q['gold_pdf_page']}.",
        f"Critical-evidence rank: {result['rank_summary']['baseline_rank']} → {result['rank_summary']['reranked_rank']}.", "",
        f"Source checklist: {q['expected_answer']}", "",
        "### Answer 1 — baseline", "", result["baseline"]["answer"], "",
        "### Answer 2 — re-ranked", "", result["reranked"]["answer"], ""]
(OUT / "evidence.md").write_text("\n".join(evidence), encoding="utf-8")
validation = dict(candidate_reuse=True, prompt_template_shared=True, source_phrases_verified=True,
                  corpus_frozen=True, embedding_normalization=True, no_answer_truncation=True,
                  timing_arithmetic=True, exploratory_queries=len(queries),
                  selected_cases=len(answers), critical_evidence_promotions=int(rank_summary.promoted_into_top3.sum()))
save_json("validation.json", validation)
display(comparison[["query_id", "category", "baseline_rank", "reranked_rank", "promoted_into_top3"]])
display(latency_summary.round(2))
print("Validation passed:", validation)
review_path = OUT / "qualitative_review.json"
if review_path.exists():
    review = json.loads(review_path.read_text())
    print("Prior manual review is saved separately; recheck it if this notebook was rerun.")

,query_id,category,baseline_rank,reranked_rank,promoted_into_top3
0,q28,keyword_trap,5,1,True
1,q02,exceptions,7,3,True
2,q11,document_specific,6,1,True
3,q06,exceptions,6,1,True
4,q13,exceptions,5,2,True
5,q29,exceptions,4,1,True
6,q25,keyword_trap,1,5,False
7,q09,negation,3,1,False


,query_id,baseline_retrieval_ms,cross_encoder_ms,reranked_retrieval_ms
0,q28,5.43,85.83,91.35
1,q02,5.35,86.19,91.53
2,q11,5.53,89.30,94.83
3,q06,5.35,86.89,92.54
4,q13,5.63,84.23,89.75
5,q29,5.75,92.04,98.17
6,q25,5.75,96.38,101.68
7,q09,5.63,88.03,93.22


Validation passed: {'candidate_reuse': True, 'prompt_template_shared': True, 'source_phrases_verified': True, 'corpus_frozen': True, 'embedding_normalization': True, 'no_answer_truncation': True, 'timing_arithmetic': True, 'exploratory_queries': 32, 'selected_cases': 8, 'critical_evidence_promotions': 6}


## 10. Student report checklist

Use `REPORT_WORKSHEET.md` to write your own concise report covering:

1. **Architecture:** independent Bi-Encoder vectors and cosine search versus joint question/passage attention in a Cross-Encoder.
2. **Observed behavior:** at least two source-verified failure cases, actual rank shifts, and whether Answer 2 fixes an omission or mistake. Include limitations and cases that did not improve.
3. **Latency:** baseline retrieval versus dense retrieval plus re-ranking, with measured units, repetition count, and hardware. Distinguish retrieval time from generation time.
4. **Scaling:** an initial dense index can search precomputed vectors; a Cross-Encoder needs a new joint forward pass for each question/passage pair. Scoring all N chunks per question costs far more than scoring the Top 10, but a two-stage system also inherits the first stage's candidate-recall limit.

Do not copy notebook explanations or generated evidence into the report as your own prose. Read the relevant PDF passages, check the answers, and explain your observations in your own words.